# Task B: Training Dynamics of the Dyck Counting Feature

这个 notebook 对应核心实验 B：在训练过程中，Transformer 什么时候形成可线性读出的 counting feature，以及这个 feature 什么时候真正转化为 Dyck next-token 行为。

当前版本先跑 smoke setting：clean-short 和 noisy-short。后续 long/sparse setting 可以用同一套脚本继续追加。

## 先读这个

**这组实验要区分两个问题。** 第一，hidden state 里是否已经有一个可线性读出的 Dyck height/counting feature；第二，模型输出 open/close 时是否真的稳定使用了这个 feature。两件事不能用同一个指标替代。

**当前最重要的结果是：**

- clean-short 的 count feature 很早出现：probe emergence step=`50`，final height R2=`0.940`。
- noisy-short 的 count feature 明显更晚：probe emergence step=`2000`，final height R2=`0.818`。
- `behavior_emergence_step` 现在按 `forced_acc >= 0.95` 判断：clean-short 在 step `50` 过阈值，noisy-short 在 step `200` 过阈值。
- raw `eval_dyck_acc` 仍然不会到 0.8，但这不表示 forced 规则没有学会。当前 Dyck 生成器有大量 free choice，clean 的 empirical oracle ceiling 约 `0.596`，noisy 的 empirical oracle ceiling 约 `0.608`。也就是说，按 raw next-token exact match 计算，0.8 不是当前数据分布下合理的成功线。
- 后面的七个追加实验主要是在排除替代解释：这个 readout 是否只是位置/随机初始化，方向是否跨 checkpoint 稳定，output head 是否沿该方向读出，以及直接干预这个 scalar 是否足以改变行为。

**一句话结论：** clean 数据下 height readout 很早变强；加入 noise 会推迟 readout。但目前证据更支持“count 是可读的表示”，还没有证明“同一个 probe direction 是直接控制 open/close 的因果旋钮”。

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'results').exists():
    ROOT = ROOT.parent
SUMMARY_DIR = ROOT / 'results' / 'dyck_counter_task_b_training_dynamics'
summary = pd.read_csv(SUMMARY_DIR / 'emergence_summary.csv')
best = pd.read_csv(SUMMARY_DIR / 'checkpoint_best_probe.csv')
behavior = pd.read_csv(SUMMARY_DIR / 'behavior_log.csv')
summary = summary.merge(
    behavior.groupby('run_key')['eval_dyck_acc'].max().rename('max_eval_dyck_acc').reset_index(),
    on='run_key',
    how='left',
)
summary

## Metric Definitions

这些指标分成三类：representation readout、behavior readout、以及二者的时间差。当前版本已经把 `behavior_emergence_step` 从 raw accuracy 阈值改成 forced accuracy 阈值；raw accuracy 只用来解释整体 next-token exact match 为什么停在 oracle ceiling 附近。

| 指标 | 定义 | 本 notebook 中如何解读 |
|---|---|---|
| `height_r2` | 用某层 hidden state 线性回归当前 Dyck height 的 held-out R2。| 主要 representation 指标；高 R2 表示 count 可线性读出。|
| `height_class_accuracy` | 把 height 当作离散 class 的 probe accuracy，用作 exact/rounded count readout 的近似。| 检查 readout 是否只是粗相关；越高说明 count 更接近离散可读。|
| `legal_next_class_accuracy` | probe 能否读出下一步 open/close 是否都 legal。| 比 height 更接近 open/close 决策所需信息，但仍是 probe，不是行为本身。|
| `eval_dyck_acc` | 模型 next-token prediction 在 Dyck token target 上的 raw accuracy。| 对当前 Markov/free-choice Dyck 不能直接用 0.8 当成功线；约 0.6 已接近当前数据的 raw 平台。|
| `max_eval_dyck_acc` | 训练日志里所有 eval step 的最大 raw Dyck next-token accuracy。| 用来确认 raw exact-match 是否贴近 oracle ceiling，而不是判断 behavior emergence。|
| `oracle_eval_dyck_acc_ceiling` | 根据 final labels 中 forced/free target 比例估计的 raw exact-match Bayes 上限：forced=1，free=0.5。| 判断 raw accuracy 的合理上限；当前 clean/noisy 都约 0.60。|
| `forced_target_fraction` / `free_target_fraction` | Dyck target 中规则唯一决定下一步的比例 / open 和 close 都合法、sampler 随机二选一的比例。| free 占比高时，整体 raw accuracy 会被 0.5 随机上限拉低。|
| `probe_emergence_step` | 第一个满足 `height_r2 >= 0.8` 且 `height_class_accuracy >= 0.5` 的 checkpoint。| 表示可读 count feature 第一次达到阈值。|
| `behavior_metric` / `behavior_threshold` | 当前用于定义 behavior emergence 的指标和阈值。| 现在是 `forced_acc >= 0.95`。|
| `behavior_emergence_step` | 第一个满足 `forced_acc >= 0.95` 的 checkpoint step。| 表示模型第一次基本学会 forced Dyck 规则。|
| `verbalization_lag` | `behavior_emergence_step - probe_emergence_step`。| 正数表示 count probe 先出现；负数表示 forced 行为先于当前 probe 阈值出现。|

## Setting Comparison

| experiment | behavior_metric | behavior_threshold | probe_emergence_step | behavior_emergence_step | verbalization_lag | early_feature_layer | best_feature_layer | final_height_r2 | final_height_class_accuracy | final_legal_next_class_accuracy | max_eval_dyck_acc | oracle_eval_dyck_acc_ceiling | forced_target_fraction | free_target_fraction | final_forced_acc | final_free_acc | final_oracle_acc | final_gap_model_minus_oracle | final_eval_dyck_acc |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| dyck_counter_task_b_clean_short_smoke | forced_acc | 0.950 | 50 | 50 | 0 | 2 | 2 | 0.940 | 0.976 | 1 | 0.623 | 0.596 | 0.191 | 0.809 | 1 | 0.501 | 0.596 | 0.001 | 0.592 |
| dyck_counter_task_b_noisy_short_smoke | forced_acc | 0.950 | 2000 | 200 | -1800 | 1 | 1 | 0.818 | 0.680 | 0.991 | 0.619 | 0.608 | 0.215 | 0.785 | 0.999 | 0.503 | 0.608 | 0.003 | 0.616 |

## Final Forced/Free Behavior Split

**这张表是解释 raw accuracy 的关键。** `eval_dyck_acc` 把 forced 和 free 混在一起；forced 位置是规则已经唯一决定下一步，free 位置是 open/close 都合法、sampler 随机二选一。

结果上，final forced acc 范围是 `0.999-1.000`，free acc 范围是 `0.501-0.503`，overall raw Dyck acc 范围是 `0.597-0.610`。因此 Task B 和 Task A 是一致的：模型已经学会 forced 规则；raw accuracy 低主要是 free step 的随机性。

| experiment | run_key | split | n | fraction | model_acc | oracle_acc | gap_model_minus_oracle |
| --- | --- | --- | --- | --- | --- | --- | --- |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | all_dyck_targets | 23552 | 1 | 0.597 | 0.596 | 0.001 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | forced | 4505 | 0.191 | 1 | 1 | 0 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | must_open | 2199 | 0.093 | 1 | 1 | 0 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | must_close | 2306 | 0.098 | 1 | 1 | 0 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | free | 19047 | 0.809 | 0.501 | 0.500 | 0.001 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | all_dyck_targets | 24171 | 1 | 0.610 | 0.608 | 0.003 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | forced | 5202 | 0.215 | 0.999 | 1 | -0.001 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | must_open | 2448 | 0.101 | 1 | 1 | 0 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | must_close | 2754 | 0.114 | 0.998 | 1 | -0.002 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | free | 18969 | 0.785 | 0.503 | 0.500 | 0.003 |

## 1. clean-short

### 1.1 实验设置

- run: `dyck_counter_task_b_clean_short_smoke/transformer_seed0`。
- task: `dyck_pairs=24`，`total_length=48`，`seq_len=48`，`repeat_prob=1.0`，`num_noise_tokens=4`。
- model: `transformer`，3-layer decoder-only Transformer，seed=`0`。
- training: `2000` steps，batch_size=`128`，lr=`0.0003`。
- checkpoints: `[0, 50, 100, 200, 500, 1000, 1500, 2000]` plus `final`。
- extraction/probes: all layers / all positions；ridge targets 为 `left/right/height`，classification targets 为 `height_class/left_right_class/legal_next_class`。

### 1.2 Emergence summary

| experiment | run_key | run_dir | probe_threshold | count_acc_threshold | behavior_metric | behavior_threshold | probe_emergence_step | stable_probe_step | behavior_emergence_step | verbalization_lag | early_feature_layer | best_feature_layer | final_height_r2 | final_height_class_accuracy | final_legal_next_class_accuracy | final_eval_dyck_acc | final_forced_acc | final_free_acc | final_oracle_acc | final_gap_model_minus_oracle | num_checkpoints_probed | max_eval_dyck_acc | oracle_eval_dyck_acc_ceiling | forced_target_fraction | free_target_fraction |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | results/dyck_counter_task_b_clean_short_smoke/transformer_seed0 | 0.800 | 0.500 | forced_acc | 0.950 | 50 | 50 | 50 | 0 | 2 | 2 | 0.940 | 0.976 | 1 | 0.592 | 1 | 0.501 | 0.596 | 0.001 | 9 | 0.623 | 0.596 | 0.191 | 0.809 |

### 1.3 Checkpoint-level best layer

| checkpoint | step | best_layer | height_r2 | height_class_accuracy | legal_next_class_accuracy | eval_dyck_acc |
| --- | --- | --- | --- | --- | --- | --- |
| step_0 | 0 | 2 | 0.754 | 0.839 | 0.966 | NA |
| step_50 | 50 | 2 | 0.809 | 0.872 | 0.978 | 0.604 |
| step_100 | 100 | 2 | 0.820 | 0.845 | 0.973 | 0.601 |
| step_200 | 200 | 2 | 0.859 | 0.851 | 0.976 | 0.610 |
| step_500 | 500 | 1 | 0.904 | 0.912 | 0.995 | 0.602 |
| step_1000 | 1000 | 1 | 0.910 | 0.961 | 0.998 | 0.622 |
| step_1500 | 1500 | 2 | 0.932 | 0.974 | 0.999 | 0.622 |
| step_2000 | 2000 | 2 | 0.940 | 0.976 | 1 | 0.592 |
| final | 2000 | 2 | 0.940 | 0.979 | 1 | 0.592 |

### 1.4 Recent behavior log

| step | train_loss | eval_loss | eval_acc | eval_dyck_acc |
| --- | --- | --- | --- | --- |
| 1450 | 0.548 | 0.551 | 0.606 | 0.606 |
| 1500 | 0.549 | 0.544 | 0.622 | 0.622 |
| 1550 | 0.554 | 0.547 | 0.599 | 0.599 |
| 1600 | 0.547 | 0.553 | 0.602 | 0.602 |
| 1650 | 0.542 | 0.552 | 0.603 | 0.603 |
| 1700 | 0.596 | 0.581 | 0.603 | 0.603 |
| 1750 | 0.554 | 0.539 | 0.618 | 0.618 |
| 1800 | 0.542 | 0.544 | 0.615 | 0.615 |
| 1850 | 0.545 | 0.547 | 0.623 | 0.623 |
| 1900 | 0.553 | 0.550 | 0.603 | 0.603 |
| 1950 | 0.545 | 0.552 | 0.600 | 0.600 |
| 2000 | 0.549 | 0.555 | 0.592 | 0.592 |

### 1.5 Training dynamics figure

![Task B training dynamics](../../figures/dyck_counter_task_b_training_dynamics/dyck_counter_task_b_clean_short_smoke_transformer_seed0.png)

### 1.6 当前结果解读

**目的。** 这里测试的是：在没有额外 noise token 的短序列里，count feature 会多早变成线性可读。

**结果。**

- count readout 第一次过阈值在 step `50`，最早可读层是 layer `2`。
- final checkpoint 的 best layer 是 layer `2`：height R2=`0.940`，height-class acc=`0.976`，legal-next acc=`1`。
- behavior emergence 用 `forced_acc >= 0.950` 判断，在 step `50` 过阈值；verbalization lag=`0`。
- final forced acc=`1`，free acc=`0.501`，model-oracle gap=`0.001`。
- raw Dyck next-token accuracy 最大值是 `0.623`，最后是 `0.592`；empirical oracle ceiling 约 `0.596`，其中 forced fraction=`0.191`，free fraction=`0.809`。
- step 0 已有 baseline：height R2=`0.754`，height-class acc=`0.839`。

**说明。**

- step 0 baseline 不能忽略。`seq_len=48`、`repeat_prob=1.0` 的生成方式会让 position/token embedding 本身带有部分 height 信息，所以不能只看“随机模型也能 probe 到一点”。真正有意义的是训练后 readout 是否超过这些 baseline、是否更稳定。
- raw Dyck accuracy 在当前 stochastic/free-choice 生成器下有约 0.6 的 oracle ceiling，所以 behavior emergence 应看 forced acc 或 oracle-normalized gap。
- 若 verbalization lag 为负，含义不是“probe 错了”，而是当前 high-R2 height probe 阈值比 forced-rule 行为更严格；模型可能先学会边界规则，再逐渐形成更完整的线性 height readout。

### 1.7 下一步检查

- 当前已经用 forced acc 定义 behavior emergence；下一步应补 oracle-normalized gap curve，检查 overall 是否贴近 Bayes ceiling。
- 补 multi-seed，确认 clean/noisy 的 probe emergence step 差异不是 seed=0 的偶然结果。
- behavior 已过 forced 阈值；重点看它相对 probe emergence 是先出现还是后出现。
- raw exact-match 受 oracle ceiling 限制；后续更适合用 layer-wise activation patch 验证 forced 规则行为是否依赖 counter readout。

## 2. noisy-short

### 2.1 实验设置

- run: `dyck_counter_task_b_noisy_short_smoke/transformer_seed0`。
- task: `dyck_pairs=24`，`total_length=48`，`seq_len=120`，`repeat_prob=0.5`，`num_noise_tokens=16`。
- model: `transformer`，3-layer decoder-only Transformer，seed=`0`。
- training: `2000` steps，batch_size=`128`，lr=`0.0003`。
- checkpoints: `[0, 50, 100, 200, 500, 1000, 1500, 2000]` plus `final`。
- extraction/probes: all layers / all positions；ridge targets 为 `left/right/height`，classification targets 为 `height_class/left_right_class/legal_next_class`。

### 2.2 Emergence summary

| experiment | run_key | run_dir | probe_threshold | count_acc_threshold | behavior_metric | behavior_threshold | probe_emergence_step | stable_probe_step | behavior_emergence_step | verbalization_lag | early_feature_layer | best_feature_layer | final_height_r2 | final_height_class_accuracy | final_legal_next_class_accuracy | final_eval_dyck_acc | final_forced_acc | final_free_acc | final_oracle_acc | final_gap_model_minus_oracle | num_checkpoints_probed | max_eval_dyck_acc | oracle_eval_dyck_acc_ceiling | forced_target_fraction | free_target_fraction |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | results/dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | 0.800 | 0.500 | forced_acc | 0.950 | 2000 | 2000 | 200 | -1800 | 1 | 1 | 0.818 | 0.680 | 0.991 | 0.616 | 0.999 | 0.503 | 0.608 | 0.003 | 9 | 0.619 | 0.608 | 0.215 | 0.785 |

### 2.3 Checkpoint-level best layer

| checkpoint | step | best_layer | height_r2 | height_class_accuracy | legal_next_class_accuracy | eval_dyck_acc |
| --- | --- | --- | --- | --- | --- | --- |
| step_0 | 0 | 2 | 0.237 | 0.224 | 0.834 | NA |
| step_50 | 50 | 2 | 0.271 | 0.241 | 0.844 | 0.588 |
| step_100 | 100 | 2 | 0.354 | 0.284 | 0.852 | 0.600 |
| step_200 | 200 | 2 | 0.400 | 0.317 | 0.868 | 0.591 |
| step_500 | 500 | 2 | 0.634 | 0.440 | 0.920 | 0.612 |
| step_1000 | 1000 | 2 | 0.761 | 0.582 | 0.974 | 0.608 |
| step_1500 | 1500 | 2 | 0.793 | 0.615 | 0.991 | 0.604 |
| step_2000 | 2000 | 1 | 0.818 | 0.680 | 0.991 | 0.616 |
| final | 2000 | 1 | 0.818 | 0.682 | 0.990 | 0.616 |

### 2.4 Recent behavior log

| step | train_loss | eval_loss | eval_acc | eval_dyck_acc |
| --- | --- | --- | --- | --- |
| 1450 | 2.538 | 2.541 | 0.247 | 0.617 |
| 1500 | 2.539 | 2.543 | 0.242 | 0.604 |
| 1550 | 2.541 | 2.539 | 0.244 | 0.609 |
| 1600 | 2.544 | 2.541 | 0.248 | 0.619 |
| 1650 | 2.535 | 2.539 | 0.247 | 0.615 |
| 1700 | 2.537 | 2.534 | 0.247 | 0.617 |
| 1750 | 2.540 | 2.541 | 0.247 | 0.617 |
| 1800 | 2.535 | 2.537 | 0.242 | 0.605 |
| 1850 | 2.532 | 2.537 | 0.247 | 0.614 |
| 1900 | 2.535 | 2.533 | 0.245 | 0.612 |
| 1950 | 2.534 | 2.539 | 0.244 | 0.610 |
| 2000 | 2.540 | 2.534 | 0.247 | 0.616 |

### 2.5 Training dynamics figure

![Task B training dynamics](../../figures/dyck_counter_task_b_training_dynamics/dyck_counter_task_b_noisy_short_smoke_transformer_seed0.png)

### 2.6 当前结果解读

**目的。** 这里测试的是：把 Dyck token 嵌入到含 noise token 的序列后，count feature 的形成是否会被推迟。

**结果。**

- count readout 第一次过阈值在 step `2000`，最早可读层是 layer `1`。
- final checkpoint 的 best layer 是 layer `1`：height R2=`0.818`，height-class acc=`0.680`，legal-next acc=`0.991`。
- behavior emergence 用 `forced_acc >= 0.950` 判断，在 step `200` 过阈值；verbalization lag=`-1800`。
- final forced acc=`0.999`，free acc=`0.503`，model-oracle gap=`0.003`。
- raw Dyck next-token accuracy 最大值是 `0.619`，最后是 `0.616`；empirical oracle ceiling 约 `0.608`，其中 forced fraction=`0.215`，free fraction=`0.785`。
- step 0 已有 baseline：height R2=`0.237`，height-class acc=`0.224`。

**说明。**

- step 0 baseline 不能忽略。`seq_len=120`、`repeat_prob=0.5` 的生成方式会让 position/token embedding 本身带有部分 height 信息，所以不能只看“随机模型也能 probe 到一点”。真正有意义的是训练后 readout 是否超过这些 baseline、是否更稳定。
- raw Dyck accuracy 在当前 stochastic/free-choice 生成器下有约 0.6 的 oracle ceiling，所以 behavior emergence 应看 forced acc 或 oracle-normalized gap。
- 若 verbalization lag 为负，含义不是“probe 错了”，而是当前 high-R2 height probe 阈值比 forced-rule 行为更严格；模型可能先学会边界规则，再逐渐形成更完整的线性 height readout。

### 2.7 下一步检查

- 当前已经用 forced acc 定义 behavior emergence；下一步应补 oracle-normalized gap curve，检查 overall 是否贴近 Bayes ceiling。
- 补 multi-seed，确认 clean/noisy 的 probe emergence step 差异不是 seed=0 的偶然结果。
- behavior 已过 forced 阈值；重点看它相对 probe emergence 是先出现还是后出现。
- raw exact-match 受 oracle ceiling 限制；后续更适合用 layer-wise activation patch 验证 forced 规则行为是否依赖 counter readout。

## 3. Additional Training Dynamics Experiments

前两节说明了 clean/noisy 的 count readout 何时出现，但还留下几个歧义。下面七个实验分别对应七个具体问题：

1. 训练更久能不能让 raw behavior 追上 probe？
2. height probe 是否只是 position 或随机初始化造成的假象？
3. 不同 checkpoint 的 probe direction 是不是同一个稳定方向？
4. output head 是否直接沿 height direction 读出 open/close？
5. 直接移动 height scalar 是否能改变模型行为？
6. 训练中的行为改善能不能泛化到更长/更稀疏的评估长度？
7. Task A 里看到的失败是否主要来自 bracket supervision density 太低？

### 3.1 Extended Training

**问题。** 训练更久以后，raw behavior 会不会追上 probe？

**做法。** 额外训练 clean/noisy 的 5k-step 版本，并和原来的 2k-step smoke run 对比。

**结果。** 2k final raw Dyck accuracy 在 `0.609-0.611`，5k final raw Dyck accuracy 在 `0.612-0.612`；5k final height R2 在 `0.861-0.953`。

**解释。** 训练更久没有把 raw Dyck accuracy 推到 0.8，说明 0.8 raw threshold 不是合适的行为成功线。同时 height probe 仍然很强，说明 representation readout 和 raw next-token behavior 需要分开报告。

| experiment | family | run_key | training_steps | max_eval_dyck_acc | final_eval_dyck_acc | final_eval_loss | final_best_layer | final_height_r2 | final_height_class_accuracy | probe_emergence_step |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| dyck_counter_task_b_clean_short_smoke | base_2k | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | 2000 | 0.623 | 0.609 | 0.548 | 2 | 0.940 | 0.979 | 50 |
| dyck_counter_task_b_noisy_short_smoke | base_2k | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | 2000 | 0.619 | 0.611 | 2.534 | 1 | 0.818 | 0.682 | 2000 |
| dyck_counter_task_b_clean_short_5k | extended_5k | dyck_counter_task_b_clean_short_5k/transformer_seed0 | 5000 | 0.622 | 0.612 | 0.544 | 1 | 0.953 | 0.981 | 1000 |
| dyck_counter_task_b_noisy_short_5k | extended_5k | dyck_counter_task_b_noisy_short_5k/transformer_seed0 | 5000 | 0.625 | 0.612 | 2.536 | 1 | 0.861 | 0.751 | 2000 |

![3.1 Extended Training](../../figures/dyck_counter_task_b_extensions/experiment_1_extended_training.png)

### 3.2 Position/Random Baseline Controls

**问题。** height probe 是否只是 position 或随机初始化带来的假象？

**做法。** 比较五种输入给同一类 ridge probe，目标都是预测同一批 token position 上的 true height：

- `position_one_hot`：只给 probe 绝对位置 one-hot，不给任何模型 hidden state。它检验 height 是否仅由序列位置分布就能猜出来。
- `position_plus_progress`：只给两个手工标量特征：normalized absolute position 和 normalized `dyck_seen`。它检验 height 是否能由位置加 Dyck 进度这种低维 schedule 信息解释。
- `random_model_hidden_step0`：给 step 0 随机初始化模型的 hidden state，使用和 final 模型相同的层。它检验随机 embedding/position encoding/未训练 Transformer 是否已经线性携带 height 信息。
- `trained_hidden_final`：给训练完成后的 final hidden state。这是我们真正关心的模型表示。
- `trained_hidden_shuffled_height`：仍给 final hidden state，但把 height label 随机打乱。这是负控；如果它也高，说明 probe 或切分流程有问题。

**结果。** trained final hidden R2=`0.810-0.941`；random step0 hidden R2=`0.236-0.767`；position-only R2≈`0.183`；shuffled-label R2≈`-0.010`。

**解释。** position 和随机初始化确实贡献了一部分可读信息，尤其 clean setting 的 step0 baseline 较高。但 trained hidden 明显强于这些 baseline，shuffled label 接近 0，说明最终 readout 不是纯伪相关。

| experiment | run_key | control | n | height_r2 | height_mae | rounded_height_accuracy |
| --- | --- | --- | --- | --- | --- | --- |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | position_one_hot | 20000 | 0.183 | 1.912 | 0.214 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | position_plus_progress | 20000 | 0.056 | 2.076 | 0.140 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | random_model_hidden_step0 | 20000 | 0.767 | 1.035 | 0.320 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | trained_hidden_final | 20000 | 0.941 | 0.499 | 0.599 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | trained_hidden_shuffled_height | 20000 | -0.010 | 2.177 | 0.127 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | position_one_hot | 20000 | 0.183 | 1.865 | 0.183 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | position_plus_progress | 20000 | 0.064 | 2.045 | 0.145 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | random_model_hidden_step0 | 20000 | 0.236 | 1.823 | 0.185 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | trained_hidden_final | 20000 | 0.810 | 0.891 | 0.370 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | trained_hidden_shuffled_height | 20000 | -0.011 | 2.152 | 0.132 |

![3.2 Position/Random Baseline Controls](../../figures/dyck_counter_task_b_extensions/experiment_2_position_random_controls.png)

### 3.3 Checkpoint Probe Transfer

**问题。** 不同 checkpoint 的 height probe direction 是不是同一个稳定方向？

**做法。** 在 checkpoint i 训练 height probe，然后冻结 probe 到 checkpoint j 上测试；如果方向稳定，off-checkpoint transfer R2 应该仍然较高。

**结果。** self-checkpoint 平均 R2≈`0.740`，off-checkpoint 平均 R2≈`-33.408`，很多早晚 checkpoint 互测为负。

**解释。** count information 可以线性读出，不代表 readout 坐标从训练一开始就稳定。尤其 step0 的 high probe 不能直接当作最终机制；训练过程中表示坐标发生了明显重排。

| experiment | mean_self_transfer_r2 | step0_to_final_r2 | final_to_step0_r2 |
| --- | --- | --- | --- |
| dyck_counter_task_b_clean_short_smoke | 0.889 | -16.353 | -19.147 |
| dyck_counter_task_b_noisy_short_smoke | 0.592 | -1.925 | -5.836 |

![3.3 Checkpoint Probe Transfer](../../figures/dyck_counter_task_b_extensions/experiment_3_probe_transfer.png)

### 3.4 Output-Head Alignment Over Time

**问题。** output head 是否直接沿 height direction 读出 open/close？

**做法。** 对每个 checkpoint 选择 height R2 最好的层，比较 probe direction 和 output head 的 `close_minus_open` 方向：一个是几何 cosine，一个是该 axis projection 与 logit margin 的相关。

**结果。** best-layer 平均 cosine≈`0.002`，axis-margin correlation≈`0.348`。

**解释。** output head 没有简单地把 probe 向量当作 close/open 权重方向。但是在数据流形上，height-axis projection 和 close-minus-open margin 有正相关，说明 count 信息可能通过更复杂的表示几何进入输出。

| experiment | checkpoint | step | layer | height_r2 | cosine_height_dir_close_minus_open | corr_height_axis_with_close_minus_open_margin |
| --- | --- | --- | --- | --- | --- | --- |
| dyck_counter_task_b_clean_short_smoke | step_0 | 0 | 2 | 0.754 | 0.092 | 0.131 |
| dyck_counter_task_b_clean_short_smoke | step_50 | 50 | 2 | 0.809 | -0.047 | 0.435 |
| dyck_counter_task_b_clean_short_smoke | step_100 | 100 | 2 | 0.820 | -0.042 | 0.431 |
| dyck_counter_task_b_clean_short_smoke | step_200 | 200 | 2 | 0.859 | -0.028 | 0.375 |
| dyck_counter_task_b_clean_short_smoke | step_500 | 500 | 1 | 0.904 | 0.002 | 0.244 |
| dyck_counter_task_b_clean_short_smoke | step_1000 | 1000 | 1 | 0.910 | 0.013 | 0.146 |
| dyck_counter_task_b_clean_short_smoke | step_1500 | 1500 | 2 | 0.932 | -0.028 | 0.348 |
| dyck_counter_task_b_clean_short_smoke | step_2000 | 2000 | 2 | 0.940 | -0.074 | 0.349 |
| dyck_counter_task_b_clean_short_smoke | final | 2000 | 2 | 0.940 | -0.074 | 0.346 |
| dyck_counter_task_b_noisy_short_smoke | step_0 | 0 | 2 | 0.237 | -0.055 | -0.062 |
| dyck_counter_task_b_noisy_short_smoke | step_50 | 50 | 2 | 0.271 | 0.421 | 0.434 |
| dyck_counter_task_b_noisy_short_smoke | step_100 | 100 | 2 | 0.354 | 0.344 | 0.504 |
| dyck_counter_task_b_noisy_short_smoke | step_200 | 200 | 2 | 0.400 | 0.273 | 0.543 |
| dyck_counter_task_b_noisy_short_smoke | step_500 | 500 | 2 | 0.634 | -0.070 | 0.407 |
| dyck_counter_task_b_noisy_short_smoke | step_1000 | 1000 | 2 | 0.761 | -0.065 | 0.411 |
| dyck_counter_task_b_noisy_short_smoke | step_1500 | 1500 | 2 | 0.793 | -0.103 | 0.335 |
| dyck_counter_task_b_noisy_short_smoke | step_2000 | 2000 | 1 | 0.818 | -0.262 | 0.451 |
| dyck_counter_task_b_noisy_short_smoke | final | 2000 | 1 | 0.818 | -0.262 | 0.444 |

![3.4 Output-Head Alignment Over Time](../../figures/dyck_counter_task_b_extensions/experiment_4_output_head_alignment.png)

### 3.4b Open/Close Branch Manifold Split

**问题。** hidden state 是否能分出 open/close 两条流形，并且这两条流形是否共用同一个 count direction？

**做法。** 对每个 checkpoint 的 best height layer 做两个版本的 branch label：

- `current_token`：当前位置已经看到的 bracket 是 open 还是 close。这个对应我们在 hidden-state 图里看到的两条 open/close 曲线。
- `next_target`：当前位置要预测的下一个 bracket target 是 open 还是 close。这个更接近 output head 的 next-token 决策，但 free step 本身有随机性。

每个版本都做三件事：先拟合 height/count direction；再比较只用 height axis 能否分类 branch；最后把 height direction 从 hidden state 投影掉，看 residual subspace 里还能不能线性分开 open/close。另外分别在 open/close 子集上拟合 height direction，计算两者 cosine，检验 count direction 是否共享。

**结果。** final checkpoint 上，`current_token` 在去掉 height direction 后的 residual branch accuracy 是 `1.000`，而只用 height axis 是 `0.567-0.569`；这说明当前 token 的 open/close 两条流形几乎完全可分，而且分离方向不只是 count axis。同一批 `current_token` 分支各自拟合出的 height direction cosine 是 `0.527-0.686`，属于中等对齐；`next_target` 的 residual branch accuracy 只有 `0.597-0.599`，但 next-open/next-close 子集的 height direction cosine 更高，为 `0.790-0.885`。

**解释。** 更准确的结论是：hidden state 里存在一个强的全局 count direction，同时 residual subspace 里还有当前 token open/close 的 branch direction。这能解释我们看到的两条曲线。但“两个分支各自的 height direction 完全相同”并不是严格成立，current-token 分支的单独拟合方向只有中等一致；可能是 current token embedding offset 和 height/parity 结构共同影响。`next_target` 不会形成同样干净的两条流形，因为 free step 的下一个 open/close 是随机采样；因此 output head alignment 不能只看 height direction，也不能期待 next-target branch 像 current-token branch 一样完全可分。

| experiment | branch_source | step | layer | height_r2 | branch_acc_full_hidden | branch_acc_height_axis_only | branch_acc_residual_after_height | height_dir_close_open_cosine | cosine_residual_branch_dir_close_minus_open_head | corr_residual_branch_axis_with_branch |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| dyck_counter_task_b_clean_short_smoke | current_token | 2000 | 2 | 0.940 | 1 | 0.569 | 1 | 0.527 | -0.030 | 1.000 |
| dyck_counter_task_b_clean_short_smoke | next_target | 2000 | 2 | 0.940 | 0.596 | 0.567 | 0.597 | 0.790 | -0.001 | 0.441 |
| dyck_counter_task_b_noisy_short_smoke | current_token | 2000 | 1 | 0.818 | 1 | 0.567 | 1 | 0.686 | -0.130 | 0.999 |
| dyck_counter_task_b_noisy_short_smoke | next_target | 2000 | 1 | 0.818 | 0.602 | 0.570 | 0.599 | 0.885 | 0.496 | 0.356 |

![3.4b Open/Close Branch Manifold Split](../../figures/dyck_counter_task_b_extensions/experiment_4b_branch_manifold_split.png)

### 3.5 Checkpoint-Wise Direct Intervention

**问题。** 直接移动 height scalar 是否能改变模型行为？

**做法。** 在每个 checkpoint 的 best layer 上，沿 height probe direction 加减若干个 axis standard deviation，再看 bracket target 上的 P(close) 和 accuracy 斜率。

**结果。** 平均 P(close) slope≈`0.0012`，accuracy slope≈`-0.0008`。

**解释。** 这个 direct hidden intervention 的效应很小，支持“probe scalar 可读但不是强因果旋钮”。注意这不是完整 forward activation patch；真正的因果验证还需要在中间层 patch 后继续跑后续层。

| experiment | checkpoint | step | layer | p_close_slope | accuracy_slope |
| --- | --- | --- | --- | --- | --- |
| dyck_counter_task_b_clean_short_smoke | step_0 | 0 | 2 | 0.002 | -0.000 |
| dyck_counter_task_b_clean_short_smoke | step_50 | 50 | 2 | -0.001 | -0.000 |
| dyck_counter_task_b_clean_short_smoke | step_100 | 100 | 2 | -0.001 | 0.000 |
| dyck_counter_task_b_clean_short_smoke | step_200 | 200 | 2 | -0.001 | -0.000 |
| dyck_counter_task_b_clean_short_smoke | step_500 | 500 | 1 | 0.000 | 0.000 |
| dyck_counter_task_b_clean_short_smoke | step_1000 | 1000 | 1 | 0.000 | 0 |
| dyck_counter_task_b_clean_short_smoke | step_1500 | 1500 | 2 | -0.001 | 0.001 |
| dyck_counter_task_b_clean_short_smoke | step_2000 | 2000 | 2 | -0.004 | 0.002 |
| dyck_counter_task_b_clean_short_smoke | final | 2000 | 2 | -0.004 | 0.002 |
| dyck_counter_task_b_noisy_short_smoke | step_0 | 0 | 2 | -0.004 | 0.003 |
| dyck_counter_task_b_noisy_short_smoke | step_50 | 50 | 2 | 0.034 | -0.006 |
| dyck_counter_task_b_noisy_short_smoke | step_100 | 100 | 2 | 0.029 | -0.004 |
| dyck_counter_task_b_noisy_short_smoke | step_200 | 200 | 2 | 0.025 | -0.003 |
| dyck_counter_task_b_noisy_short_smoke | step_500 | 500 | 2 | -0.005 | 0.000 |
| dyck_counter_task_b_noisy_short_smoke | step_1000 | 1000 | 2 | -0.007 | 0.001 |
| dyck_counter_task_b_noisy_short_smoke | step_1500 | 1500 | 2 | -0.012 | 0.002 |
| dyck_counter_task_b_noisy_short_smoke | step_2000 | 2000 | 1 | -0.015 | -0.006 |
| dyck_counter_task_b_noisy_short_smoke | final | 2000 | 1 | -0.015 | -0.006 |

![3.5 Checkpoint-Wise Direct Intervention](../../figures/dyck_counter_task_b_extensions/experiment_5_checkpoint_intervention.png)

### 3.6 Length Generalization Over Training

**问题。** 训练中的行为改善能否泛化到更长/更稀疏的评估长度？

**做法。** 固定 checkpoint，额外评估多个 `eval_seq_len` 和 `repeat_prob`，让 Dyck token 在更长上下文中更稀疏。

**结果。** final checkpoint 的 Dyck accuracy 范围是 `0.514-0.608`；最短 eval 长度均值≈`0.608`，最长 eval 长度均值≈`0.515`。

**解释。** 更长/更稀疏的评估会压低行为，说明短分布上的训练改善不等价于稳健长度泛化。这和 Task A 的 density 结论一致。

| experiment | run_key | checkpoint | step | eval_setting | eval_seq_len | eval_repeat_prob | accuracy | dyck_accuracy | num_examples |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_0 | 0 | seq48_p1.000 | 48 | 1 | 0.147 | 0.147 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_0 | 0 | seq120_p0.400 | 120 | 0.400 | 0.168 | 0.187 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_0 | 0 | seq500_p0.096 | 500 | 0.096 | 0.150 | 0.213 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_0 | 0 | seq1000_p0.048 | 1000 | 0.048 | 0.144 | 0.222 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_500 | 500 | seq48_p1.000 | 48 | 1 | 0.602 | 0.602 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_500 | 500 | seq120_p0.400 | 120 | 0.400 | 0.206 | 0.515 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_500 | 500 | seq500_p0.096 | 500 | 0.096 | 0.052 | 0.514 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_500 | 500 | seq1000_p0.048 | 1000 | 0.048 | 0.030 | 0.516 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_1000 | 1000 | seq48_p1.000 | 48 | 1 | 0.609 | 0.609 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_1000 | 1000 | seq120_p0.400 | 120 | 0.400 | 0.210 | 0.526 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_1000 | 1000 | seq500_p0.096 | 500 | 0.096 | 0.051 | 0.523 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_1000 | 1000 | seq1000_p0.048 | 1000 | 0.048 | 0.026 | 0.515 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_2000 | 2000 | seq48_p1.000 | 48 | 1 | 0.608 | 0.608 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_2000 | 2000 | seq120_p0.400 | 120 | 0.400 | 0.215 | 0.537 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_2000 | 2000 | seq500_p0.096 | 500 | 0.096 | 0.052 | 0.528 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | step_2000 | 2000 | seq1000_p0.048 | 1000 | 0.048 | 0.026 | 0.515 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | final | 2000 | seq48_p1.000 | 48 | 1 | 0.608 | 0.608 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | final | 2000 | seq120_p0.400 | 120 | 0.400 | 0.215 | 0.537 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | final | 2000 | seq500_p0.096 | 500 | 0.096 | 0.052 | 0.528 | 256 |
| dyck_counter_task_b_clean_short_smoke | dyck_counter_task_b_clean_short_smoke/transformer_seed0 | final | 2000 | seq1000_p0.048 | 1000 | 0.048 | 0.026 | 0.515 | 256 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | step_0 | 0 | seq120_p0.500 | 120 | 0.500 | 0.074 | 0.115 | 256 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | step_0 | 0 | seq500_p0.096 | 500 | 0.096 | 0.054 | 0.118 | 256 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | step_0 | 0 | seq1000_p0.048 | 1000 | 0.048 | 0.050 | 0.119 | 256 |
| dyck_counter_task_b_noisy_short_smoke | dyck_counter_task_b_noisy_short_smoke/transformer_seed0 | step_500 | 500 | seq120_p0.500 | 120 | 0.500 | 0.243 | 0.608 | 256 |

只显示前 24 行；完整表在 `results/dyck_counter_task_b_training_dynamics/`。

![3.6 Length Generalization Over Training](../../figures/dyck_counter_task_b_extensions/experiment_6_length_generalization.png)

### 3.7 Bracket-Density Curve

**问题。** Task A/Task B 里的 long sparse 失败，主要来自长上下文本身，还是来自 bracket supervision 太稀疏？

**做法。** 固定 `seq_len=2000`，只扫 bracket token 数量；这样长度不变，只改变 Dyck supervision density。

**结果。** forced accuracy 第一次超过 0.8 约在 `48` 个 bracket token；free accuracy 接近 0.5 平台约在 `64` 个 bracket token。height R2 在很低 density 时也能保持可读，但 behavior 到 48-64 bracket tokens 后才明显恢复。

**解释。** 失败的主要来源不是单纯 `seq_len=2000`，而是 Dyck target 在 next-token loss 中过于稀疏。这解释了为什么 tiny_extreme_long 可以有较高 height probe，却没有稳定 behavior。

| source | bracket_tokens | density | dyck_accuracy | oracle_accuracy | forced_accuracy | free_accuracy | height_r2 | legal_next_accuracy |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| tiny_extreme_long | 20 | 0.010 | 0.113 | 0.680 | 0.263 | 0.005 | 0.804 | 0.970 |
| sparse_len2000_b24 | 24 | 0.012 | 0.052 | 0.663 | 0.187 | 0.004 | 0.822 | 0.965 |
| sparse_len2000_b28 | 28 | 0.014 | 0.080 | 0.647 | 0.221 | 0.009 | 0.815 | 0.975 |
| sparse_len2000_b32 | 32 | 0.016 | 0.087 | 0.640 | 0.236 | 0.007 | 0.793 | 0.867 |
| sparse_len2000_b34 | 34 | 0.017 | 0.074 | 0.634 | 0.447 | 0.007 | 0.777 | 0.971 |
| sparse_len2000_b36 | 36 | 0.018 | 0.160 | 0.633 | 0.602 | 0.013 | 0.783 | 0.966 |
| sparse_len2000_b40 | 40 | 0.020 | 0.283 | 0.625 | 0.777 | 0.043 | 0.755 | 0.859 |
| sparse_len2000_b44 | 44 | 0.022 | 0.290 | 0.621 | 0.760 | 0.042 | 0.753 | 0.871 |
| sparse_len2000_b48 | 48 | 0.024 | 0.208 | 0.612 | 0.826 | 0.054 | 0.779 | 0.894 |
| sparse_len2000_b56 | 56 | 0.028 | 0.377 | 0.606 | 0.927 | 0.137 | 0.764 | 0.903 |
| sparse_len2000_b64 | 64 | 0.032 | 0.504 | 0.600 | 0.977 | 0.463 | 0.732 | 0.897 |
| sparse_len2000_b80 | 80 | 0.040 | 0.591 | 0.590 | 0.962 | 0.492 | 0.781 | 0.902 |
| sparse_len2000_b100 | 100 | 0.050 | 0.565 | 0.578 | 0.988 | 0.503 | 0.685 | 0.897 |
| sparse_len2000_b200 | 200 | 0.100 | 0.566 | 0.556 | 0.990 | 0.499 | 0.733 | 0.938 |
| extreme_long | 400 | 0.200 | 0.555 | 0.539 | 0.986 | 0.501 | 0.730 | 0.958 |

![3.7 Bracket-Density Curve](../../figures/dyck_counter_task_b_extensions/experiment_7_density_curve.png)

## 4. Overall Interpretation

**目前这批结果支持的结论：**

1. Transformer hidden state 中确实会形成可线性读出的 Dyck height/counting feature。clean-short 很早出现，noisy-short 明显延后。
2. forced behavior emergence 已经可以用 `forced_acc >= 0.95` 定义。当前 clean 在 step 50 过阈值，noisy 在 step 200 过阈值；raw Dyck next-token accuracy 仍只适合和 oracle ceiling 比较。
3. position 和随机初始化能解释一部分早期 readout，尤其 clean setting；但 trained hidden 明显更强，shuffled-label 控制接近 0，所以最终 probe 不是纯假象。
4. noisy setting 里 forced behavior 先于 high-R2 height probe 出现，说明边界规则行为和完整线性 height readout 不是同一个时间点。probe direction 在训练中并不总是稳定，output head 也没有简单对齐这个方向。因此“height 可线性读出”不能直接推出“这个线性方向就是模型的因果控制变量”。
5. 长上下文失败更像是 bracket supervision density 问题：固定 `seq_len=2000` 时，bracket token 从 20 增加到 48-64 后 forced/free behavior 才明显恢复。

**下一步最值得补的是：**

- 把 oracle-normalized gap 也做成 checkpoint curve，和 forced behavior emergence 一起报告。
- 做 multi-seed，确认 clean/noisy emergence timing 和 density threshold 是否稳定。
- 做真正的 layer-wise activation patch：在中间层 patch donor activation 后继续 forward，而不是只在 final hidden 上沿 probe direction 做 direct-logit intervention。

In [ ]:
# 可选：重新汇总并生成 notebook
# !python scripts/task_b_training_dynamics.py --experiments dyck_counter_task_b_clean_short_smoke dyck_counter_task_b_noisy_short_smoke
# !python scripts/task_b_training_dynamics_extensions.py --device cuda --eval-examples 256 --eval-batch-size 32 --max-rows 20000
# !python scripts/write_task_b_training_dynamics_notebook.py